
# AhorraMás Web Scraper

This notebook collects product and price information from AhorraMás using Selenium and BeautifulSoup.

### How it works

1. Opens the AhorraMás website and accepts cookies.
2. Searches for the product terms defined in `SEARCH_TERMS`.
3. Loads additional products using the "Más resultados" button.
4. Extracts product names, IDs, prices, units, URLs, and timestamps.
5. Reviews the collected RAW data and reports incomplete searches.
6. Saves the current RAW collection and a dated historical RAW snapshot.

### Data storage

- Current RAW: `data/raw/ahorramas_products_raw.csv`
- Historical RAW: `data/history/ahorramas/YYYY-MM-DD_raw.csv`

Deduplication, standardization, classification, and historical price processing are handled by the central pipeline.

TEST_MODE allows extraction tests without saving files.



In [ ]:

import time
import re
from pathlib import Path
from datetime import datetime
from urllib.parse import urljoin, quote

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException


HOME_URL = "https://www.ahorramas.com/"

SEARCH_TERMS = [
    "leche",
    "huevos",
    "manzana",
    "plátano",
    "tomate",
    "arroz",
    "pasta",
    "pan",
    "pollo",
    "carne picada",
    "salmón",
    "yogur",
    "queso",
    "agua",
    "zumo",
    "atún",
    "garbanzos",
    "aceite",
    "detergente",
    "papel higiénico",
]

# --------------------------------------------------
# TEST MODE
# --------------------------------------------------

TEST_MODE = False
TEST_TERMS = ["leche", "yogur", "garbanzos", "detergente"]

TERMS_TO_SCRAPE = TEST_TERMS if TEST_MODE else SEARCH_TERMS

print("Search terms to scrape:", TERMS_TO_SCRAPE)



# --------------------------------------------------
# PROJECT DIRECTORIES
# --------------------------------------------------

def find_project_root():
    current = Path.cwd().resolve()

    for path in (current, *current.parents):
        if path.name == "supermarket_project":
            return path

    if current.name == "scrapers":
        return current.parent

    raise FileNotFoundError("Could not locate the project root.")


BASE_DIR = find_project_root()

RAW_DIR = BASE_DIR / "data" / "raw"
HISTORY_DIR = BASE_DIR / "data" / "history" / "ahorramas"

RAW_PATH = RAW_DIR / "ahorramas_products_raw.csv"

RUN_DATE = datetime.now().strftime("%Y-%m-%d")
RAW_HISTORY_PATH = HISTORY_DIR / f"{RUN_DATE}_raw.csv"

print("Project:", BASE_DIR)
print("Collection date:", RUN_DATE)
print("Current RAW:", RAW_PATH)
print("Historical RAW:", RAW_HISTORY_PATH)


# --------------------------------------------------
# SELENIUM DRIVER CONFIGURATION
# --------------------------------------------------

def start_ahorramas_driver():
    options = Options()

    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option(
        "excludeSwitches", ["enable-automation"]
    )
    options.add_experimental_option(
        "useAutomationExtension", False
    )
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=options)

    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', "
        "{get: () => undefined})"
    )

    driver.get(HOME_URL)

    return driver


def accept_ahorramas_cookies(driver):
    xpaths = [
        "//button[contains(., 'Aceptar')]",
        "//button[contains(., 'Permitir')]",
        "//button[@id='onetrust-accept-btn-handler']"
    ]

    for xpath in xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xpath)

            for btn in buttons:
                if btn.is_displayed():
                    btn.click()
                    print("AhorraMás cookies accepted.")
                    return

        except Exception:
            pass

    print("Cookie banner not found or already accepted.")


# --------------------------------------------------
# PRODUCT SEARCH AND PAGINATION
# --------------------------------------------------

def count_product_links(driver):
    # Count unique links matching potential product URLs
    links = driver.find_elements(
        By.CSS_SELECTOR,
        'a[href*=".html"], a[class*="product"], div.product-tile a'
    )

    hrefs = {
        link.get_attribute("href")
        for link in links
        if link.get_attribute("href")
    }

    return len(hrefs)


def search_ahorramas(driver, term):
    encoded_term = quote(term)
    search_url = f"https://www.ahorramas.com/search?q={encoded_term}"

    print(f"Navigating to: {search_url}")

    driver.get(search_url)
    time.sleep(4)


def load_all_ahorramas_products(driver, max_clicks=100):

    print("Loading all available products...")

    clicks = 0
    selector = "button.more[data-url]"

    while clicks < max_clicks:

        # Find the "More results" button in the DOM
        buttons = driver.find_elements(By.CSS_SELECTOR, selector)

        enabled_buttons = [
            button for button in buttons
            if button.is_displayed() and button.is_enabled()
        ]

        if not enabled_buttons:

            # Distinguish hidden buttons from completed pagination
            if buttons:
                print("More results button exists but is not visible.")
                return count_product_links(driver), "button_not_visible"

            print("No more results button found.")
            return count_product_links(driver), "complete"

        button = enabled_buttons[0]

        # Scroll directly to the button instead of the page bottom
        driver.execute_script(
            "arguments[0].scrollIntoView({"
            "behavior: 'instant', block: 'center'"
            "});",
            button
        )

        time.sleep(0.5)

        previous_count = count_product_links(driver)
        previous_url = button.get_attribute("data-url")

        try:
            # Load the next batch of products
            button.click()

            # Wait until new products or updated pagination are detected
            WebDriverWait(driver, 20).until(
                lambda d: (
                    count_product_links(d) > previous_count
                    or not any(
                        b.is_displayed()
                        and b.get_attribute("data-url") == previous_url
                        for b in d.find_elements(
                            By.CSS_SELECTOR, selector
                        )
                    )
                )
            )

        except TimeoutException:
            print("Timeout loading additional products.")
            return count_product_links(driver), "timeout"

        except Exception as exc:
            print(f"Error: {type(exc).__name__}: {exc}")
            return count_product_links(driver), "error"

        clicks += 1

        current_count = count_product_links(driver)

        print(
            f"Click {clicks}: "
            f"{previous_count} -> {current_count} links"
        )

    print("Maximum number of clicks reached.")
    return count_product_links(driver), "max_clicks_reached"


# --------------------------------------------------
# PRODUCT DATA EXTRACTION
# --------------------------------------------------

def parse_ahorramas_price(card, text):
    """
    Return (price, price_raw) for one product card.

    The site renders the price as
    <span class="sales"><span class="value" content="0.83">0<span class="decimal-part">,83€</span>
    so get_text(" ") yields "0 ,83€". Read the structured content attribute of the
    selling price (not the strike-through list price shown before it on offers).
    """
    value = card.select_one(".sales .value[content]")

    if value is not None:
        try:
            price = float(value["content"])
            price_raw = re.sub(r"\s+", "", value.get_text())
            return price, price_raw
        except ValueError:
            pass

    # Fallback for older markup: join "0 ,83" back together first
    joined = re.sub(r"(\d)\s+([.,]\d)", r"\1\2", text)

    price_match = re.search(r"(\d+(?:[.,]\d{1,2})?)\s*€", joined)

    if price_match:
        return (
            float(price_match.group(1).replace(",", ".")),
            price_match.group(0),
        )

    return None, None


def extract_ahorramas_products(driver, searched_item):

    soup = BeautifulSoup(driver.page_source, "html.parser")

    cards = soup.select(
        'div.product-tile, li.grid-tile, div.grid-tile, '
        'div[class*="product-tile"]'
    )

    if not cards:
        cards = soup.select('div[class*="product"], article')

    records = []
    seen_urls = set()

    scraped_at = datetime.now().isoformat(timespec="seconds")

    for card in cards:
        text = card.get_text(" ", strip=True)

        # Identify the product URL
        link_elem = card.find(
            "a", href=re.compile(r"\.html|/p/")
        )

        if not link_elem:
            link_elem = card.find("a", href=True)

        if not link_elem:
            continue

        href = link_elem.get("href")

        if not href or "search" in href or "cart" in href:
            continue

        url = urljoin(HOME_URL, href)

        if url in seen_urls:
            continue

        seen_urls.add(url)

        # Extract product name
        name = ""

        for tag in ["h3", "h2", "h4", "a", "span", "div"]:
            elem = card.find(
                tag,
                class_=re.compile("title|name|desc|link", re.I)
            )

            if elem:
                txt = elem.get_text(" ", strip=True)

                if len(txt) > 4 and "€" not in txt:
                    name = txt
                    break

        if not name:
            link_text = link_elem.get_text(" ", strip=True)

            if len(link_text) > 4:
                name = link_text

        if not name:
            continue

        # Extract the selling price (discounted price when on offer)
        price, price_raw = parse_ahorramas_price(card, text)

        # Discard only missing, zero or negative prices
        if price is None or price <= 0:
            continue

        # Extract product ID from URL
        id_match = re.search(
            r"/(\d+)\.html|/p/(\d+)", url
        )

        product_id = (
            id_match.group(1) or id_match.group(2)
            if id_match else None
        )

        # --------------------------------------------------
        # EXTRACT QUANTITY AND UNIT PRICE
        # --------------------------------------------------

        unit = None
        price_per_unit = None
        quantity_val = None

        name_lower = name.lower()

        # Detect liters and milliliters
        l_match = re.search(
            r"(\d+(?:[.,]\d+)?)\s*(?:l|litros?)\b",
            name_lower
        )

        ml_match = re.search(
            r"(\d+)\s*ml\b",
            name_lower
        )

        # Detect kilograms and grams
        kg_match = re.search(
            r"(\d+(?:[.,]\d+)?)\s*(?:kg|k)\b",
            name_lower
        )

        g_match = re.search(
            r"(\d+)\s*(?:g|gr)\b",
            name_lower
        )

        # Detect individual units
        u_match = re.search(
            r"(\d+)\s*(?:u|uds|unidades?)\b",
            name_lower
        )

        if l_match:
            quantity_val = float(
                l_match.group(1).replace(",", ".")
            )
            unit = "LITRO"

        elif ml_match:
            quantity_val = float(ml_match.group(1)) / 1000.0
            unit = "LITRO"

        elif kg_match:
            quantity_val = float(
                kg_match.group(1).replace(",", ".")
            )
            unit = "KILO"

        elif g_match:
            quantity_val = float(g_match.group(1)) / 1000.0
            unit = "KILO"

        elif u_match:
            quantity_val = float(u_match.group(1))
            unit = "UNIDAD"

            if quantity_val == 12 and "docena" in name_lower:
                unit = "DOCENA"

        # Calculate price per unit when quantity is available
        if quantity_val and quantity_val > 0:
            price_per_unit = round(
                price / quantity_val, 4
            )

        # Store extracted product information
        records.append({
            "searched_item": searched_item,
            "supermarket": "AhorraMás",
            "product_id": product_id,
            "name": name,
            "quantity_raw": None,
            "price_raw": price_raw,
            "price": price,
            "price_per_unit_raw": None,
            "price_per_unit": price_per_unit,
            "unit": unit,
            "url": url,
            "scraped_at": scraped_at,
        })

    df = pd.DataFrame(records)

    print(
        f"Products extracted for '{searched_item}': {len(df)}"
    )

    return df


Search terms to scrape: ['leche', 'yogur', 'garbanzos', 'detergente']
Project: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project
Collection date: 2026-11-20
Current RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\ahorramas_products_raw.csv
Historical RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\history\ahorramas\2026-11-20_raw.csv


In [2]:

# --------------------------------------------------
# AHORRAMAS COLLECTION WITH DEFERRED RETRIES
# --------------------------------------------------

# Initial collection + two additional retry passes.
MAX_ADDITIONAL_ATTEMPTS = 2
MAX_COLLECTION_PASSES = 1 + MAX_ADDITIONAL_ATTEMPTS

# Only successful searches are stored.
successful_results = {}

# Keep exactly one final summary per search term.
summary_records = {}

driver = None


# --------------------------------------------------
# COLLECTION PASSES
# --------------------------------------------------

try:

    for pass_number in range(1, MAX_COLLECTION_PASSES + 1):

        # Retry only terms that have not succeeded.
        pending_terms = [
            term
            for term in TERMS_TO_SCRAPE
            if term not in successful_results
        ]

        if not pending_terms:
            print("\nAll searches completed successfully.")
            break

        print("\n" + "=" * 70)
        print(
            f"COLLECTION PASS "
            f"{pass_number}/{MAX_COLLECTION_PASSES}"
        )
        print("Pending searches:", len(pending_terms))
        print("=" * 70)

        for i, term in enumerate(pending_terms, start=1):

            print(
                f"\n[{i}/{len(pending_terms)}] "
                f"AhorraMás - {term} "
                f"(attempt {pass_number})"
            )

            started = time.monotonic()

            status = "error"
            error_message = None
            loaded_count = None
            extracted_count = 0

            try:

                # --------------------------------------
                # START OR REUSE SELENIUM
                # --------------------------------------

                if driver is None:

                    print("Starting Chrome driver...")

                    driver = start_ahorramas_driver()

                    time.sleep(3)

                    accept_ahorramas_cookies(driver)

                # --------------------------------------
                # SEARCH AND LOAD PRODUCTS
                # --------------------------------------

                search_ahorramas(driver, term)

                loaded_count, load_status = (
                    load_all_ahorramas_products(driver)
                )

                # Do not accept partially loaded searches.
                if load_status != "complete":

                    raise RuntimeError(
                        f"Product loading finished with "
                        f"status '{load_status}'. "
                        f"Links detected: {loaded_count}."
                    )

                # --------------------------------------
                # EXTRACT PRODUCTS
                # --------------------------------------

                df_term = extract_ahorramas_products(
                    driver,
                    term
                )

                extracted_count = len(df_term)

                if df_term.empty:

                    raise RuntimeError(
                        "No valid products were extracted."
                    )

                # --------------------------------------
                # STORE SUCCESSFUL SEARCH
                # --------------------------------------

                # Replace the result for this term.
                # Never append results from failed attempts.
                successful_results[term] = df_term

                status = "complete"

                print(
                    f"SUCCESS: {extracted_count} "
                    f"products collected."
                )

                time.sleep(2)

            except Exception as exc:

                error_message = (
                    f"{type(exc).__name__}: {exc}"
                )

                print(
                    f"FAILED: {term} - {error_message}"
                )

                print(
                    "The search will be retried "
                    "after the remaining terms."
                )

                # Discard partial results from this attempt.
                extracted_count = 0

                # Restart Chrome for the next search.
                # This also helps recover from broken sessions.
                if driver is not None:

                    try:
                        driver.quit()
                    except Exception:
                        pass

                    driver = None

            elapsed = round(
                time.monotonic() - started,
                2
            )

            previous_seconds = summary_records.get(
                term, {}
            ).get("seconds", 0)

            # Replace the previous summary for this term.
            summary_records[term] = {
                "searched_item": term,
                "status": status,
                "products_loaded": loaded_count,
                "rows_extracted": extracted_count,
                "attempts": pass_number,
                "seconds": round(
                    previous_seconds + elapsed,
                    2
                ),
                "error": error_message,
            }

        # ----------------------------------------------
        # PASS SUMMARY
        # ----------------------------------------------

        remaining_terms = [
            term
            for term in TERMS_TO_SCRAPE
            if term not in successful_results
        ]

        print("\nPASS SUMMARY")
        print(
            "Successful searches:",
            len(successful_results)
        )
        print(
            "Remaining failures:",
            len(remaining_terms)
        )

        if (
            remaining_terms
            and pass_number < MAX_COLLECTION_PASSES
        ):

            print("\nSearches to retry:")

            for term in remaining_terms:
                print(f"  - {term}")

            # Give the website a short recovery pause.
            time.sleep(10)


finally:

    # Always close Chrome at the end.
    if driver is not None:

        try:
            driver.quit()
        except Exception:
            pass


# --------------------------------------------------
# BUILD FINAL RAW DATA
# --------------------------------------------------

# Keep each successful search exactly once.
# Preserve the original search-term order.

all_results = [
    successful_results[term]
    for term in TERMS_TO_SCRAPE
    if term in successful_results
]

df_ahorramas = (
    pd.concat(
        all_results,
        ignore_index=True
    )
    if all_results
    else pd.DataFrame()
)


# --------------------------------------------------
# BUILD FINAL SUMMARY
# --------------------------------------------------

# One summary row per search term.
summary = [
    summary_records[term]
    for term in TERMS_TO_SCRAPE
]

df_summary = pd.DataFrame(summary)


# --------------------------------------------------
# FINAL COLLECTION REPORT
# --------------------------------------------------

print("\n" + "=" * 70)
print("FINAL AHORRAMAS COLLECTION SUMMARY")
print("=" * 70)

display(df_summary)

successful_count = int(
    df_summary["status"].eq("complete").sum()
)

recovered_searches = df_summary[
    df_summary["status"].eq("complete")
    & df_summary["attempts"].gt(1)
]

failed_searches = df_summary[
    df_summary["status"].ne("complete")
]

print("\nExpected searches:", len(TERMS_TO_SCRAPE))
print("Successful searches:", successful_count)
print("Recovered after retry:", len(recovered_searches))
print("Final failures:", len(failed_searches))
print("Total RAW rows:", len(df_ahorramas))


# --------------------------------------------------
# REPORT RECOVERED SEARCHES
# --------------------------------------------------

if not recovered_searches.empty:

    print("\nRECOVERED SEARCHES:")

    display(
        recovered_searches[
            [
                "searched_item",
                "attempts",
                "rows_extracted",
                "seconds"
            ]
        ]
    )


# --------------------------------------------------
# REPORT FINAL FAILURES
# --------------------------------------------------

if not failed_searches.empty:

    print("\nWARNING: Some searches remain incomplete.")

    display(
        failed_searches[
            [
                "searched_item",
                "status",
                "attempts",
                "error"
            ]
        ]
    )

    print(
        "Incomplete collections must not be published."
    )

else:

    print("\nAll searches completed successfully.")
    print("RAW data is ready for validation and saving.")



COLLECTION PASS 1/3
Pending searches: 4

[1/4] AhorraMás - leche (attempt 1)
Starting Chrome driver...
AhorraMás cookies accepted.
Navigating to: https://www.ahorramas.com/search?q=leche
Loading all available products...
Click 1: 44 -> 63 links
Click 2: 63 -> 83 links
Click 3: 83 -> 103 links
Click 4: 103 -> 123 links
Click 5: 123 -> 143 links
Click 6: 143 -> 163 links
Click 7: 163 -> 183 links
Click 8: 183 -> 194 links
No more results button found.
Products extracted for 'leche': 166
SUCCESS: 166 products collected.

[2/4] AhorraMás - yogur (attempt 1)
Navigating to: https://www.ahorramas.com/search?q=yogur
Loading all available products...
Click 1: 43 -> 63 links
Click 2: 63 -> 83 links
Click 3: 83 -> 103 links
Click 4: 103 -> 123 links
Click 5: 123 -> 143 links
Click 6: 143 -> 163 links
Click 7: 163 -> 183 links
Click 8: 183 -> 203 links
Click 9: 203 -> 216 links
No more results button found.
Products extracted for 'yogur': 183
SUCCESS: 183 products collected.

[3/4] AhorraMás - ga

,searched_item,status,products_loaded,rows_extracted,attempts,seconds,error
0,leche,complete,194,166,1,205.20,None
1,yogur,complete,216,183,1,280.70,None
2,garbanzos,complete,46,22,1,28.27,None
3,detergente,complete,89,65,1,105.38,None



Expected searches: 4
Successful searches: 4
Recovered after retry: 0
Final failures: 0
Total RAW rows: 436

All searches completed successfully.
RAW data is ready for validation and saving.


In [3]:
# --------------------------------------------------
# PREPARE RAW DATA
# --------------------------------------------------

if df_ahorramas.empty:
    print("No raw data collected.")
else:
    print("Raw data ready for processing.")
    print("Total rows:", len(df_ahorramas))
    print("Search terms:", df_ahorramas["searched_item"].nunique())


Raw data ready for processing.
Total rows: 436
Search terms: 4


In [4]:

# --------------------------------------------------
# VALIDATE AND SAVE RAW DATA
# --------------------------------------------------

import os
import tempfile

if TEST_MODE:
    print("TEST MODE: no files will be saved.")

else:
    # --------------------------------------------------
    # VALIDATE COLLECTED DATA
    # --------------------------------------------------

    if df_ahorramas.empty:
        raise RuntimeError(
            "No products collected. No files saved."
        )

    required_columns = [
        "searched_item",
        "supermarket",
        "product_id",
        "name",
        "quantity_raw",
        "price_raw",
        "price",
        "price_per_unit_raw",
        "price_per_unit",
        "unit",
        "url",
        "scraped_at",
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df_ahorramas.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"Missing RAW columns: {missing_columns}"
        )

    if df_summary.empty:
        raise RuntimeError(
            "No search results were recorded. No files saved."
        )

    valid_names = (
        df_ahorramas["name"].notna()
        & df_ahorramas["name"].astype(str).str.strip().ne("")
    )

    numeric_prices = pd.to_numeric(
        df_ahorramas["price"], errors="coerce"
    )

    valid_prices = numeric_prices.notna() & numeric_prices.gt(0)

    if not valid_names.any() or not valid_prices.any():
        raise RuntimeError(
            "No valid product names or selling prices found. "
            "No files saved."
        )

    # Preserve every RAW row, including repeated products and
    # records that may require validation in the central pipeline.
    invalid_rows = int((~(valid_names & valid_prices)).sum())

    if invalid_rows:
        print(
            f"WARNING: {invalid_rows} rows have missing names "
            "or invalid prices. RAW values will be preserved."
        )

    # --------------------------------------------------
    # CHECK COLLECTION COMPLETENESS
    # --------------------------------------------------

    expected_terms = set(SEARCH_TERMS)
    attempted_terms = set(df_summary["searched_item"])

    all_terms_attempted = (
        len(df_summary) == len(SEARCH_TERMS)
        and attempted_terms == expected_terms
    )

    all_searches_complete = (
        df_summary["status"].eq("complete").all()
    )

    all_searches_have_results = (
        df_summary["rows_extracted"].gt(0).all()
    )

    collection_complete = bool(
        all_terms_attempted
        and all_searches_complete
        and all_searches_have_results
    )

    incomplete_searches = df_summary[
        (df_summary["status"] != "complete")
        | (df_summary["rows_extracted"] <= 0)
    ]


    # --------------------------------------------------
    # PREVENT INCOMPLETE DATA FROM BEING SAVED
    # --------------------------------------------------

    if not collection_complete:

        print("\nERROR: Incomplete collection detected.")

        if not incomplete_searches.empty:

            display(
                incomplete_searches[
                    [
                        "searched_item",
                        "status",
                        "attempts",
                        "rows_extracted",
                        "error"
                    ]
                ]
            )

        raise RuntimeError(
            "AhorraMás collection is incomplete after "
            "all retry attempts. No RAW files were saved."
        )


    # --------------------------------------------------
    # PROTECT HISTORICAL SNAPSHOTS
    # --------------------------------------------------

    if RAW_HISTORY_PATH.exists():
        raise FileExistsError(
            "A RAW historical snapshot already exists "
            f"for {RUN_DATE}: {RAW_HISTORY_PATH}. "
            "No files were modified."
        )

    # --------------------------------------------------
    # PREPARE CSV FILES
    # --------------------------------------------------

    def prepare_raw_temp(destination):
        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        fd, temp_name = tempfile.mkstemp(
            prefix=".ahorramas_",
            suffix=".tmp",
            dir=destination.parent
        )

        try:
            with os.fdopen(
                fd,
                "w",
                encoding="utf-8-sig",
                newline=""
            ) as file:
                df_ahorramas.to_csv(
                    file,
                    index=False
                )

            return Path(temp_name)

        except Exception:
            Path(temp_name).unlink(missing_ok=True)
            raise

    history_temp = None
    current_temp = None

    try:
        history_temp = prepare_raw_temp(RAW_HISTORY_PATH)
        current_temp = prepare_raw_temp(RAW_PATH)

        # Save history first, without overwriting an existing date.
        with history_temp.open("rb") as source:
            with RAW_HISTORY_PATH.open("xb") as target:
                while True:
                    chunk = source.read(1024 * 1024)
                    if not chunk:
                        break
                    target.write(chunk)

        print("Saved historical RAW:", RAW_HISTORY_PATH)

        # Update the latest RAW only after saving history.
        os.replace(current_temp, RAW_PATH)

        print("Saved current RAW:", RAW_PATH)

    finally:
        if history_temp is not None:
            history_temp.unlink(missing_ok=True)

        if current_temp is not None:
            current_temp.unlink(missing_ok=True)

    # --------------------------------------------------
    # COLLECTION SUMMARY
    # --------------------------------------------------

    print("\nCOLLECTION SUMMARY")
    print("Date:", RUN_DATE)
    print(
        "Status:",
        "complete" if collection_complete else "partial"
    )
    print("RAW rows:", len(df_ahorramas))
    print(
        "Distinct product IDs:",
        df_ahorramas["product_id"].nunique()
    )
    print("Searches attempted:", len(df_summary))
    print("Expected searches:", len(SEARCH_TERMS))
    print("Rows requiring validation:", invalid_rows)

    print("\nSaved outputs:")
    print(RAW_PATH)
    print(RAW_HISTORY_PATH)


TEST MODE: no files will be saved.
